In [1]:
!pip install matplotlib scipy pandas cvxpy tqdm seaborn openai cvxpy[glpk] polarix -q

zsh:1: no matches found: cvxpy[glpk]


In [2]:
import pickle
import sys
import time                                                                                                                                    
from pathlib import Path
from collections import Counter, defaultdict                                                                                                   
from itertools import combinations

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.optimize import linprog
import pandas as pd
sys.path.insert(0, str(Path.cwd()))

from src.iterative_game_analysis.metagame import MetaGame

from visuals.visualize_analysis import DISPLAY_NAMES, STRATEGY_ORDER
from evaluation.curb_analysis import *
from evaluation.bootstrap_analysis import load_all_games, compute_payoff_matrix_from_games 
from src.iterative_game_analysis.full_analysis import load_crossplay_to_dataframe                                                              
from src.iterative_game_analysis.bootstrap import Bootstrap                                                                                    

/Users/gabesmithline/.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/matplotlib-71xvjj0b because there was an issue with the default path (/Users/gabesmithline/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


## CURB Sets as Explainability for Meta-Game Analysis

The project extends the LLM bargaining meta-game paper, which reports a single MENE equilibrium, bootstrap confidence intervals, and a best-response graph on a 10-strategy empirical game of LLM negotiators, into an iterative framework for interpreting larger meta-games. As the strategy library grows to include more models, prompting variants, and heuristic baselines, the payoff matrix becomes harder to read directly and single-equilibrium summaries become harder to interpret. We propose CURB sets as an explainability layer on top of standard meta-game analysis. Rather than relying only on one equilibrium over the full strategy library, we identify strategically closed subgames in which every best response remains inside the set, yielding self-sustaining strategic ecologies that can be analyzed on their own terms. To validate the idea, we study synthetic games generated from the Candogan potential–harmonic–nonstrategic decomposition and test when CURB structure is most informative. Our hypothesis is that CURB is most useful in games with substantial cyclic or conflicting incentives, precisely where equilibrium summaries are hardest to explain. These synthetic experiments serve as a proof of concept before applying the pipeline to empirical LLM bargaining games, where the current instance appears relatively concentrated but larger and more competitive libraries should make the benefits of CURB-based explanation much clearer.

**Slogan:** *CURB for structural explanation, equilibrium for characterization within each structure.*

---

### Motivation: Fragility of Single-Equilibrium Summaries

Recent work (Liu et al., 2025, ICLR) has shown that in general-sum evaluation games, equilibrium summaries selected via Shannon-entropy regularization can be sensitive to redundant actions — duplicating a strategy can shift the selected equilibrium for representational rather than substantive strategic reasons. This underscores the fragility of single-equilibrium summaries in rich empirical games: even if we keep using a solver like MENE, we should not assume that one selected equilibrium fully captures the game's strategic structure.

Our goal is different but complementary: rather than proposing a new equilibrium selector, we introduce a **CURB-based structural layer** that decomposes the empirical game into strategically coherent equilibrium-bearing subgames, allowing equilibrium multiplicity to be interpreted through ecologies rather than only through one selected solution.

The conceptual bridge:
- **Their motivation**: one selected equilibrium may be unstable or misleading because the selection criterion itself can be fragile to redundancy.
- **Our motivation**: therefore, we want a structural lens that shows whether the game contains multiple coherent strategic regions that a single selected equilibrium may hide.

If equilibrium selection is fragile, then strategy importance should not be summarized only by support under one selected equilibrium. Appearance frequency, conditional support, and regret across CURB-induced equilibria provide a more **ecology-aware notion of strategic importance**.

*Note: We do not claim that CURB solves the redundancy problem directly. It provides a decomposition / interpretability layer that is less tied to any one equilibrium selector. Their solver work is about how to pick a better equilibrium-based rating summary; our CURB work is about how to understand the game when any one equilibrium summary may be incomplete.*

---

### Core Role of CURB

CURB sets are not a replacement for Nash equilibrium. They are an **interpretability / decomposition tool**. A single MENE can hide:
- multiplicity of equilibrium-bearing regions,
- modular strategic structure,
- alternative ecologies with different welfare / fairness properties.

CURB exposes this structure. Instead of saying *"here is the game's equilibrium,"* we say *"here are the game's equilibrium-bearing strategic ecologies, and here is what behavior looks like inside each one."*

---

### Theoretical Foundation

#### Equilibrium Guarantee (Basu & Weibull, 1991)

Each minimal CURB set contains the support of at least one mixed Nash equilibrium:

$$\forall C \in \mathcal{C}_{\min}(G),\ \exists \sigma \in \mathrm{NE}(G) \text{ such that } \operatorname{supp}(\sigma) \subseteq C$$

**Important:** The converse is *not* true in general. Not every Nash equilibrium support is contained in a minimal CURB set — Basu and Weibull give counterexamples (e.g., perfect equilibria whose support falls outside any minimal CURB set). So the correct framing is:

- **What is true:** Each minimal CURB set guarantees at least one equilibrium lives inside it.
- **What is not true:** Every equilibrium is captured by some minimal CURB set.

This means minimal CURB sets identify **strategically coherent, equilibrium-bearing subgames**. By solving each one, we recover a structured subset of the game's equilibrium possibilities — not a complete cover of all equilibria, but a principled decomposition into self-contained ecologies that each sustain equilibrium behavior.

#### Disjoint Ecologies

If $C_1, \dots, C_K$ are pairwise disjoint minimal CURB sets, then each contains the support of at least one Nash equilibrium. The game possesses at least $K$ equilibrium-bearing strategic ecologies.

- MENE answers: *"what is one equilibrium summary?"*
- Minimal disjoint CURB sets answer: *"what are distinct strategic worlds that each sustain at least one equilibrium?"*

#### Restricted-Game Implication

If $C$ is a CURB set and $\sigma^*$ is a Nash equilibrium of the restricted game $G|_C$, then $\sigma^*$ is also a Nash equilibrium of the full game $G$. This is what makes CURB useful as a decomposition layer: restrict → solve → the solution is valid in the full game.

#### Computational Tractability

For two-player normal-form games, following Benisch, Davis, and Sandholm (JAIR 2010), all minimal CURB sets can be found in **polynomial time**. So in the bimatrix empirical-game setting, CURB is a tractable preprocessing / decomposition step.

---

### Formal Pipeline

Let the full empirical game be $G = (S, u)$ symmetric with $S = \{1, \dots, n\}$. Let $\mathcal{C}_{\min}(G) = \{C^{(1)}, \dots, C^{(K)}\}$ be the minimal CURB sets. Let $\Phi$ be a common solver (e.g., MENE).

1. **Full-game solution:** $\sigma^{\text{full}} := \Phi(G)$
2. **Restricted ecology solution:** $\sigma^{(k)} := \Phi(G|_{C^{(k)}})$ for each $k$, with $\sigma^{(k)}_i = 0$ for $i \notin C^{(k)}$
3. **Lift** each $\sigma^{(k)}$ back to the full strategy space (pad zeros outside $C^{(k)}$) → $\widetilde{\sigma}^{(k)}$
4. **Compare** each $\widetilde{\sigma}^{(k)}$ to $\sigma^{\text{full}}$ in the ambient game

Optionally attach nonnegative weights $w_k$ to ecologies with $\sum_k w_k = 1$ (uniform, bootstrap frequencies, or salience scores).

**Two evaluation perspectives:**
- **Internal coherence**: evaluate the equilibrium inside $G|_C$ — how the ecology behaves on its own terms
- **External competitiveness**: evaluate the lifted solution in $G$ — how compelling the ecology remains in the ambient game

---

### Strategy-Level Ecology Profile

For each strategy $i \in S$, we define a profile $\Pi_i = (A_i, \bar{s}_i, \bar{s}_i^{\text{cond}}, \bar{r}_i, \bar{r}_i^{\text{appear}})$ that characterizes its role across the CURB ecology family.

#### 1. Appearance Frequency

$$A_i := \sum_{k=1}^K w_k \, \mathbf{1}\{\sigma_i^{(k)} > 0\}$$

How often strategy $i$ appears with positive support across CURB-induced equilibria.

#### 2. Unconditional Average Support

$$\bar{s}_i := \sum_{k=1}^K w_k \, \sigma_i^{(k)}$$

Average equilibrium mass assigned to strategy $i$ across all ecologies.

#### 3. Conditional Support Given Appearance

$$\bar{s}_i^{\text{cond}} := \frac{\sum_{k=1}^K w_k \, \sigma_i^{(k)} \, \mathbf{1}\{\sigma_i^{(k)} > 0\}}{A_i}$$

How much support $i$ gets in the ecologies where it is actually active.

#### 4. Pointwise Regret Against an Ecology

$$r_i^{(k)} := \max_{j \in S} u(j, \sigma^{(k)}) - u(i, \sigma^{(k)})$$

Loss from playing $i$ instead of a best response to $\sigma^{(k)}$. If $i \in \operatorname{supp}(\sigma^{(k)})$ and $\sigma^{(k)}$ is an exact NE, then $r_i^{(k)} = 0$.

#### 5. Average Regret Across Ecologies

$$\bar{r}_i := \sum_{k=1}^K w_k \, r_i^{(k)}$$

How poorly strategy $i$ performs, on average, against the ecology family. Low $\bar{r}_i$ = broadly competitive; high $\bar{r}_i$ = broadly misaligned.

#### 6. Conditional Regret Given Appearance

$$\bar{r}_i^{\text{appear}} := \frac{\sum_{k=1}^K w_k \, r_i^{(k)} \, \mathbf{1}\{\sigma_i^{(k)} > 0\}}{A_i}$$

When strategy $i$ is used in a CURB-induced equilibrium, how close is it to a best response there?

#### 7. Ecology Specialization Gap

$$\text{SpecGap}_i := \bar{r}_i - \bar{r}_i^{\text{appear}}$$

- Large $\text{SpecGap}_i$: strategy $i$ is a **specialist** — much better in its own ecologies than globally
- Small $\text{SpecGap}_i$: strategy $i$ is a **generalist** — performs similarly everywhere

#### Strategy Typology

| Type | Signature | Interpretation |
|------|-----------|----------------|
| Rare but heavy | $A_i$ small, $\bar{s}_i^{\text{cond}}$ large | Niche strategy that matters a lot when it matters |
| Common but light | $A_i$ large, $\bar{s}_i^{\text{cond}}$ small | Background stabilizer, appears often but rarely dominates |
| Specialist | $\bar{r}_i$ large, $\bar{r}_i^{\text{appear}}$ small | Weak globally, strong in its own ecologies |
| Generalist | $\bar{r}_i$ small, $A_i$ large | Broadly competitive across the ecology family |

---

### Ecology-Level Diagnostics

#### Full-Game Regret per Ecology

$$R_k := R(\widetilde{\sigma}^{(k)}; G)$$

| Statistic | Definition | Interpretation |
|-----------|-----------|----------------|
| Average CURB regret | $R_{\text{avg}} = \frac{1}{K} \sum_k R_k$ | Mean regret across ecologies |
| Worst-case | $R_{\max} = \max_k R_k$ | Least stable ecology |
| CURB-cover regret | $R_{\text{cover}} = \min_k R_k$ | Best low-regret explanation of the full game |

#### Ecology-Weighted Regret

$$R^{(k)} := \sum_{i \in S} \sigma_i^{(k)} \, r_i^{(k)}, \qquad \bar{R}^{\text{CURB}} := \sum_{k=1}^K w_k \, R^{(k)}$$

If all lifted CURB equilibria are exact full-game NE, then $R^{(k)} = 0$ for all $k$. In approximate or bootstrap settings, this serves as a **sanity / robustness diagnostic**. The more substantive signal across ecologies comes from welfare, fairness, support/entropy, bootstrap stability, and distance from the full-game MENE.

---

### Bootstrap Pipeline

Per bootstrap sample $b$:
1. Resample payoff observations → bootstrap game $G^{(b)}$
2. Compute minimal CURB sets in $G^{(b)}$
3. Solve the full game $G^{(b)}$ under the chosen solver → $\sigma_{\text{full}}^{(b)}$
4. Solve each CURB-restricted game $G^{(b)}|_{C_k^{(b)}}$ → $\sigma_k^{(b)}$
5. Lift each restricted solution back to full space
6. Compute full-game metrics for each lifted ecology solution

This gives uncertainty over both the **existence/composition** of ecologies and their **evaluation**. Summaries include:
- How often a given CURB set appears across bootstraps
- Probability that strategies $i$ and $j$ co-occur in the same minimal CURB (co-membership matrix)
- Distribution of welfare/regret per ecology
- Whether the full-game solver consistently lands in one ecology
- Bootstrap-weighted welfare / fairness / regret

**Matching across bootstraps:** Since CURB sets may shift between samples, we use a strategy co-membership matrix (probability $i,j$ share a minimal CURB) as the most stable summary.

---

### Distilled Summary

- Use minimal CURB sets as strategic ecologies
- Each minimal CURB set contains at least one Nash equilibrium support (but not all NE are necessarily captured)
- In two-player empirical games, minimal CURB sets can be enumerated in polynomial time
- Solve each ecology with the same solver as the full game
- Lift those solutions back to full space and compare them
- Profile each strategy by its ecology appearance, support, regret, and specialization
- Use regret as a consistency check; welfare/fairness/stability as the substantive interpretability signal

*Minimal CURB sets provide a tractable family of strategically coherent, equilibrium-bearing subgames. By solving each CURB-restricted game under the same solution concept used for the full empirical game, and then evaluating those lifted solutions in the ambient game, we obtain an interpretable decomposition that recovers a structured subset of the game's equilibrium possibilities — refining a single-equilibrium summary such as MENE. For each strategy, the ecology profile $(A_i, \bar{s}_i, \bar{s}_i^{\text{cond}}, \bar{r}_i, \bar{r}_i^{\text{appear}})$ distinguishes strategies that are equilibrium-rare but heavily weighted when active from those that are equilibrium-common but weakly supported, and separates ecology-specific specialists from broadly competitive generalists.*

---

### Synthetic Game Generation

Games are parameterized along the **potential–harmonic spectrum** (Candogan et al. 2011):
- **Potential** component: coordination/alignment — pure NE exist
- **Harmonic** component: conflict/cyclic — no pure NE generically
- **Nonstrategic** component: payoff shifts that don't affect best responses

In [3]:
USE_SYNTHETIC = False   # <-- flip to False to use real bargaining data
N_STRATEGIES = 15      # number of strategies (scales to 100+)

if USE_SYNTHETIC:
    from evaluation.curb_analysis import (
        _run_brute_force, compute_cbr,
        find_minimal_curb_sets_via_closure,
        find_all_curb_sets_via_closure,
    )

    def make_potential_component(n, rng):
        """Potential game: pairwise interaction + strategy quality.
        Every pair has a unique interaction term. Scales via vectorized ops.
        """
        phi = rng.normal(0, 10, size=n)
        W = rng.normal(0, 8, size=(n, n))
        W = (W + W.T) / 2
        return phi[:, None] + phi[None, :] + W

    def make_harmonic_component(n, rng):
        """Harmonic game: multiple isolated RPS cycles + cross-group interactions.

        Vectorized for large n. Groups of size ~4, within-group cyclic dominance,
        weaker cross-group cycles, penalty for out-of-group play.
        """
        n_groups = max(2, n // 4)
        M = np.zeros((n, n))
        indices = rng.permutation(n)
        splits = np.array_split(indices, n_groups)

        # Build group membership array
        group_of = np.zeros(n, dtype=int)
        position_in_group = np.zeros(n, dtype=int)
        group_size = np.zeros(n, dtype=int)
        for g_idx, group in enumerate(splits):
            for pos, i in enumerate(group):
                group_of[i] = g_idx
                position_in_group[i] = pos
                group_size[i] = len(group)

        # Within-group: cyclic dominance (vectorized)
        within_strength = 25
        same_group = group_of[:, None] == group_of[None, :]
        pos_i = position_in_group[:, None]
        pos_j = position_in_group[None, :]
        k = group_size[:, None]  # group size for row player

        # Cyclic: (pos_i - pos_j) % k == 1 → win, reverse → loss
        diff = (pos_i - pos_j) % k
        wins = same_group & (diff == 1) & (pos_i != pos_j)
        losses = same_group & (((pos_j - pos_i) % k) == 1) & (pos_i != pos_j)
        neutral = same_group & ~wins & ~losses & (np.arange(n)[:, None] != np.arange(n)[None, :])

        noise_wins = rng.normal(0, 3, size=(n, n))
        noise_losses = rng.normal(0, 3, size=(n, n))
        noise_neutral = rng.normal(0, 5, size=(n, n))

        M += wins * (within_strength + noise_wins)
        M -= losses * (within_strength + noise_losses)
        M += neutral * noise_neutral

        # Cross-group: neighboring groups have cyclic advantage
        cross_strength = 8
        neighbor_group = (group_of[:, None] + 1) % n_groups
        is_neighbor = (~same_group) & (group_of[None, :] == neighbor_group)
        is_neighbor_rev = (~same_group) & (group_of[:, None] == (group_of[None, :] + 1) % n_groups)

        noise_cross = rng.normal(0, 4, size=(n, n))
        M += is_neighbor * (cross_strength + noise_cross)
        M -= is_neighbor_rev * (cross_strength + rng.normal(0, 4, size=(n, n)))

        # Other cross-group: small random
        other_cross = (~same_group) & (~is_neighbor) & (~is_neighbor_rev)
        M += other_cross * rng.normal(0, 4, size=(n, n))

        # Out-of-group penalty
        penalty_noise = rng.normal(0, 2, size=(n, n))
        M -= (~same_group) * (12 + penalty_noise)

        return M

    def make_nonstrategic_component(n, rng):
        """Nonstrategic: M[i,j] = c[j]. Doesn't affect best responses."""
        return np.tile(rng.normal(0, 5, size=n), (n, 1))

    def generate_game(alpha_pot=0.0, alpha_harm=1.0, alpha_ns=0.2,
                      n=10, seed=42, baseline=50):
        """Game = baseline + α_pot·Potential + α_harm·Harmonic + α_ns·Nonstrategic"""
        rng = np.random.default_rng(seed)
        P = make_potential_component(n, rng)
        H = make_harmonic_component(n, rng)
        N = make_nonstrategic_component(n, rng)
        M = baseline + alpha_pot * P + alpha_harm * H + alpha_ns * N
        return M, {'potential': P, 'harmonic': H, 'nonstrategic': N}

    # --- Generate games along the potential–harmonic spectrum ---
    n_strategies = N_STRATEGIES
    spectrum = {
        #'Pure Potential':  {'alpha_pot': 1.0, 'alpha_harm': 0.0, 'alpha_ns': 0.2},
        #'Pot-Heavy Mix':   {'alpha_pot': 0.7, 'alpha_harm': 0.3, 'alpha_ns': 0.2},
        #'Balanced':        {'alpha_pot': 0.5, 'alpha_harm': 0.5, 'alpha_ns': 0.2},
        #'Harm-Heavy Mix':  {'alpha_pot': 0.3, 'alpha_harm': 0.7, 'alpha_ns': 0.2},
        'Pure Harmonic':   {'alpha_pot': 0.0, 'alpha_harm': 1.0, 'alpha_ns': 0.2},
    }

    use_brute_force = (n_strategies <= 20)

    print(f"Generating {n_strategies}-strategy games ({'brute force' if use_brute_force else 'closure-based'} CURB)")
    print(f"{'Game Type':<20} {'# CURB':>7} {'# Minimal':>10} {'Min Sizes':>30}")
    print("-" * 70)

    spectrum_results = {}
    for label, params in spectrum.items():
        M, components = generate_game(**params, n=n_strategies, seed=42)
        minimal = find_minimal_curb_sets_via_closure(M, n_strategies)
        min_sizes = sorted([len(s) for s in minimal])

        if use_brute_force:
            all_curb, _ = _run_brute_force(M, n_strategies)
        else:
            all_curb = find_all_curb_sets_via_closure(M, n_strategies)

        spectrum_results[label] = {
            'payoff': M, 'components': components,
            'all_curb': all_curb, 'minimal': minimal,
        }
        sizes_str = str(min_sizes) if len(str(min_sizes)) <= 30 else f"{len(minimal)} sets, range [{min(min_sizes)}-{max(min_sizes)}]"
        print(f"{label:<20} {len(all_curb):>7} {len(minimal):>10} {sizes_str:>30}")

    # --- Select active game for downstream analysis ---
    active_game = 'Pure Harmonic'
    avg_payoff = spectrum_results[active_game]['payoff']
    strategy_names = [f"S{i}" for i in range(n_strategies)]

    # Generate welfare/fairness matrices
    nw_matrix = np.sqrt(np.clip(avg_payoff, 1, None) * np.clip(avg_payoff.T, 1, None))
    nw_matrix = nw_matrix / nw_matrix.max() * 100

    surplus = avg_payoff - np.percentile(avg_payoff, 25)
    nw_plus_matrix = np.sqrt(np.clip(surplus, 0.1, None) * np.clip(surplus.T, 0.1, None))
    nw_plus_matrix = nw_plus_matrix / nw_plus_matrix.max() * 100

    payoff_diff = np.abs(avg_payoff - avg_payoff.T)
    ef1_matrix = np.exp(-payoff_diff / 30)
    ef1_plus_matrix = np.exp(-payoff_diff / 20)

    matrices = {
        "payoff": avg_payoff,
        "nw": nw_matrix,
        "nw_plus": nw_plus_matrix,
        "ef1": ef1_matrix,
        "ef1_plus": ef1_plus_matrix,
    }

    print(f"\n--- Active game: {active_game} ({n_strategies} strategies) ---")
    print(f"Minimal CURB sets: {len(spectrum_results[active_game]['minimal'])}")
    for s in sorted(spectrum_results[active_game]['minimal'], key=len):
        print(f"  |{len(s)}|: {sorted(s)}")
    print(f"Payoff range: [{avg_payoff.min():.1f}, {avg_payoff.max():.1f}]")
    print(f"Payoff mean: {avg_payoff.mean():.1f}, std: {avg_payoff.std():.1f}")

In [4]:
import json
# if not USE_SYNTHETIC:
#     # crossplay_dir = Path.cwd() / "data" / "crossplay"   
#     # strategy_names = ["walk", "tough", "soft",
#     #                     "openai_5.2_none", "openai_5.2_low", "ef1_bargainer", "ppo", "psro", "nfsp"] 
#     # f = load_crossplay_to_dataframe(crossplay_dir, strategy_names, raw_utility=True)  
#     # print(f"Loaded {len(df)} rows, columns: {list(df.columns)}")

# else:
#     print("Using synthetic game — skipping real data load.")

import gc                                                                                                                                                                 
                                                                                                                                                                        
crossplay_dir = Path.cwd() / "data" / "crossplay"
strategy_names = ["walk", "tough", "soft",
                "openai_5.2_none", "openai_5.2_low", "ef1_bargainer", "nfsp", "ppo", "psro", "mappo"]

# Load one matchup at a time, append to list, then free memory
dfs = []
for si in strategy_names:
    for sj in strategy_names:
        games_path = crossplay_dir / f"{si}_p1_vs_{sj}_p2" / "games.json"
        if not games_path.exists():
            print(f"  MISSING: {si} vs {sj}")
            continue

        print(f"  Loading {si} vs {sj}...", end=" ")
        for attempt in range(3):
            try:
                with open(games_path, "rb") as f:                                                                                                                                 
                    raw = f.read()
                if len(raw) == 0:
                    raise OSError("Empty file read (possibly iCloud evicted)")
                data = json.loads(raw)
                del raw
                break
            except (TimeoutError, OSError, json.JSONDecodeError) as e:
                print(f"retry {attempt+1} ({e})...", end=" ")
                del raw
                gc.collect()
                import time; time.sleep(5)
        else:
            print("FAILED")
            continue

        games = data["games"]
        n_games = len(games)

        payoff_p1 = np.zeros(n_games)
        payoff_p2 = np.zeros(n_games)
        batna_p1 = np.zeros(n_games)
        batna_p2 = np.zeros(n_games)
        utilities_p1 = np.zeros((n_games, 3))
        utilities_p2 = np.zeros((n_games, 3))
        ef1 = np.full(n_games, np.nan)

        accept_indices = []
        alloc_p1_list = []
        alloc_p2_list = []
        util_p1_list = []
        util_p2_list = []

        for idx, game in enumerate(games):
            outcome = game["outcome"]
            payoff_p1[idx] = outcome["payoff_p1"]
            payoff_p2[idx] = outcome["payoff_p2"]
            batna_p1[idx] = outcome["batna_p1"]
            batna_p2[idx] = outcome["batna_p2"]
            if outcome.get("utilities_p1"):
                utilities_p1[idx] = outcome["utilities_p1"]
                utilities_p2[idx] = outcome["utilities_p2"]
            if outcome["result"] == "accept":
                if outcome.get("utilities_p1") and outcome.get("allocation_p1"):
                    accept_indices.append(idx)
                    util_p1_list.append(outcome["utilities_p1"])
                    util_p2_list.append(outcome["utilities_p2"])
                    alloc_p1_list.append(outcome["allocation_p1"])
                    alloc_p2_list.append(outcome["allocation_p2"])

        del data, games

        # EF1 computation
        from src.iterative_game_analysis.full_analysis import _is_ef1_vectorized, _exists_ef1_beating_batnas, ITEM_QUANTITIES
        if accept_indices:
            ef1_results = _is_ef1_vectorized(
                np.array(util_p1_list), np.array(util_p2_list),
                np.array(alloc_p1_list), np.array(alloc_p2_list),
            )
            ef1[accept_indices] = ef1_results.astype(float)

        # Raw utility space
        max_p1 = np.sum(utilities_p1 * ITEM_QUANTITIES, axis=1)
        max_p2 = np.sum(utilities_p2 * ITEM_QUANTITIES, axis=1)
        raw_pay_p1 = payoff_p1 * max_p1
        raw_pay_p2 = payoff_p2 * max_p2
        raw_bat_p1 = batna_p1 * max_p1
        raw_bat_p2 = batna_p2 * max_p2

        # EF1+
        outcome_beats = (raw_pay_p1 > raw_bat_p1) & (raw_pay_p2 > raw_bat_p2)
        ef1_rational_exists = _exists_ef1_beating_batnas(
            utilities_p1, utilities_p2, raw_bat_p1, raw_bat_p2,
        )
        ef1_plus = np.full(n_games, np.nan)
        is_accept = np.zeros(n_games, dtype=bool)
        is_accept[accept_indices] = True
        rational_accept = is_accept & ef1_rational_exists
        ef1_plus[rational_accept] = (
            ef1[rational_accept].astype(bool) & outcome_beats[rational_accept]
        ).astype(float)

        # Role-symmetrized rows
        forward = pd.DataFrame({
            "policy_i": si, "policy_j": sj,
            "payoff_i": raw_pay_p1, "payoff_j": raw_pay_p2,
            "batna_i": raw_bat_p1, "batna_j": raw_bat_p2,
            "ef1": ef1, "ef1_plus": ef1_plus,
        })
        reverse = pd.DataFrame({
            "policy_i": sj, "policy_j": si,
            "payoff_i": raw_pay_p2, "payoff_j": raw_pay_p1,
            "batna_i": raw_bat_p2, "batna_j": raw_bat_p1,
            "ef1": ef1, "ef1_plus": ef1_plus,
        })
        dfs.append(forward)
        dfs.append(reverse)

        del forward, reverse, payoff_p1, payoff_p2, utilities_p1, utilities_p2
        gc.collect()
        print(f"{n_games} games")

df = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()
print(f"\nLoaded {len(df)} total rows ({len(df) // 2} games x 2 directions)")

  Loading walk vs walk... 300000 games
  Loading walk vs tough... 300000 games
  Loading walk vs soft... 300000 games
  Loading walk vs openai_5.2_none... 300000 games
  Loading walk vs openai_5.2_low... 300000 games
  Loading walk vs ef1_bargainer... 300000 games
  Loading walk vs nfsp... 300000 games
  Loading walk vs ppo... 300000 games
  Loading walk vs psro... 300000 games
  Loading walk vs mappo... 300000 games
  Loading tough vs walk... 300000 games
  Loading tough vs tough... 300000 games
  Loading tough vs soft... 20000 games
  Loading tough vs openai_5.2_none... 1000 games
  Loading tough vs openai_5.2_low... 1000 games
  Loading tough vs ef1_bargainer... 20000 games
  Loading tough vs nfsp... 20000 games
  Loading tough vs ppo... 20000 games
  Loading tough vs psro... 20000 games
  Loading tough vs mappo... 20000 games
  Loading soft vs walk... 300000 games
  Loading soft vs tough... 20000 games
  Loading soft vs soft... 20000 games
  Loading soft vs openai_5.2_none... 1000 

In [5]:
if not USE_SYNTHETIC:
    # Build the average matrices (no bootstrap, just the point estimate)
    boot = Bootstrap(df=df, n_samples=1, seed=42, policies=strategy_names)
    matrices = boot._build_all_matrices(df, strategy_names)

    avg_payoff = matrices["payoff"]       # 10x10, used for equilibrium solving + UW
    nw_matrix = matrices["nw"]            # 10x10, Nash welfare per matchup
    nw_plus_matrix = matrices["nw_plus"]  # 10x10, Nash welfare on advantages
    ef1_matrix = matrices["ef1"]          # 10x10, EF1 frequency per matchup
    ef1_plus_matrix = matrices["ef1_plus"]# 10x10, EF1+ frequency per matchup       

print("Empirical Meta-Game (avg payoff):\n")
header = "".join(f"{s[:8]:>10}" for s in strategy_names)
print(f"{'':>10}{header}")
print("-" * (10 + 10 * len(strategy_names)))
for i, name in enumerate(strategy_names):
    row = "".join(f"{avg_payoff[i, j]:>10.2f}" for j in range(len(strategy_names)))
    print(f"{name[:8]}{row}")

Empirical Meta-Game (avg payoff):

                walk     tough      soft  openai_5  openai_5  ef1_barg      nfsp       ppo      psro     mappo
--------------------------------------------------------------------------------------------------------------
walk    303.81    303.81    303.81    303.81    303.81    303.81    303.81    303.81    303.81    303.81
tough    303.81    303.81    579.17    330.69    324.55    304.62    325.16    319.51    317.67    320.49
soft    303.81     50.43    303.59    230.81    229.89    310.96    287.51    197.45    165.56    141.60
openai_5    303.81    298.73    446.43    381.55    382.81    372.72    357.85    358.69    343.24    346.93
openai_5    303.81    302.70    438.95    383.37    382.31    377.30    368.21    364.88    364.36    348.93
ef1_barg    303.81    305.08    394.19    367.14    363.18    362.15    352.80    346.18    345.92    348.70
nfsp    303.81    301.64    412.38    369.82    366.16    363.91    356.00    352.84    347.91    34

In [6]:
import networkx as nx

# Compute best response graph                                                                                                                                             
n = len(strategy_names)
br_graph = nx.DiGraph()                                                                                                                                                   
br_graph.add_nodes_from(strategy_names)

for j in range(n):
    # Best response to pure strategy j
    best_payoff = np.max(avg_payoff[:, j])
    for i in range(n):
        if np.isclose(avg_payoff[i, j], best_payoff, atol=0.5):
            br_graph.add_edge(strategy_names[j], strategy_names[i])

# Identify sinks (self-loops = best response to self)
sinks = [s for s in strategy_names if br_graph.has_edge(s, s)]

# Layout
pos = nx.spring_layout(br_graph, seed=42, k=2)

fig, ax = plt.subplots(figsize=(10, 8))

# Color nodes by type
colors = []
for s in br_graph.nodes():
    if s in sinks:
        colors.append('#e74c3c')  # red = sink
    elif s in ['ppo', 'psro', 'mappo', 'nfsp']:
        colors.append('#3498db')  # blue = RL
    elif s in ['openai_5.2_none', 'openai_5.2_low', 'ef1_bargainer']:
        colors.append('#e6550d')  # orange = LLM
    else:
        colors.append('#bbbbbb')  # grey = heuristic

# Draw
nx.draw_networkx_nodes(br_graph, pos, node_color=colors, node_size=800, ax=ax)
nx.draw_networkx_labels(br_graph, pos, font_size=8, ax=ax)

# Separate self-loops from other edges for styling
self_loops = [(u, v) for u, v in br_graph.edges() if u == v]
other_edges = [(u, v) for u, v in br_graph.edges() if u != v]

nx.draw_networkx_edges(br_graph, pos, edgelist=other_edges,
                        arrows=True, arrowsize=20, edge_color='#555555',
                        connectionstyle='arc3,rad=0.1', ax=ax)
nx.draw_networkx_edges(br_graph, pos, edgelist=self_loops,
                        arrows=True, arrowsize=20, edge_color='#e74c3c',
                        connectionstyle='arc3,rad=0.3', ax=ax)

ax.set_title(f'Best Response Graph (sinks: {", ".join(sinks)})')
ax.axis('off')
plt.tight_layout()
plt.savefig('best_response_graph_non_avg.png', dpi=150, bbox_inches='tight')
plt.close()

# Print BR table
print(f'\n{"Against":<25} {"Best Response":}')
print('-' * 50)
for j in range(n):
    brs = [strategy_names[i] for i in range(n)
            if np.isclose(avg_payoff[i, j], np.max(avg_payoff[:, j]), atol=0.5)]
    print(f'{strategy_names[j]:<25} {", ".join(brs)}')


Against                   Best Response
--------------------------------------------------
walk                      walk, tough, soft, openai_5.2_none, openai_5.2_low, ef1_bargainer, nfsp, ppo, psro, mappo
tough                     ppo
soft                      tough
openai_5.2_none           openai_5.2_low
openai_5.2_low            mappo
ef1_bargainer             openai_5.2_low
nfsp                      openai_5.2_low
ppo                       ppo
psro                      psro
mappo                     mappo


In [7]:
from evaluation.curb_analysis import _run_brute_force, METRIC_KEYS, find_minimal_curb_sets_via_closure, find_all_curb_sets_via_closure


n = len(strategy_names)

# if USE_SYNTHETIC and n > 20:
#     # Already computed in the synthetic cell — reuse
#     all_curb = spectrum_results[active_game]['all_curb']
#     minimal_curb = spectrum_results[active_game]['minimal']
# elif n <= 20:
#     all_curb, minimal_curb = _run_brute_force(avg_payoff, n)
# else:
#     # Large real game — use closure-based
#     minimal_curb = find_minimal_curb_sets_via_closure(avg_payoff, n)
#     all_curb = find_all_curb_sets_via_closure(avg_payoff, n)

all_curb, minimal_curb = _run_brute_force(avg_payoff, n)
print(f"Total CURB sets: {len(all_curb)}, Minimal: {len(minimal_curb)}")
for s in sorted(minimal_curb, key=len):
    print(f"  |{len(s)}|: {sorted(s)}")

Total CURB sets: 40, Minimal: 3
  |1|: [7]
  |1|: [8]
  |1|: [9]


In [8]:
for S in all_curb:
    names = sorted([strategy_names[i] for i in S])
    print(f"  |S|={len(S)}: {names}")

  |S|=1: ['ppo']
  |S|=1: ['psro']
  |S|=1: ['mappo']
  |S|=2: ['ppo', 'tough']
  |S|=2: ['mappo', 'openai_5.2_low']
  |S|=2: ['ppo', 'psro']
  |S|=2: ['mappo', 'psro']
  |S|=3: ['ppo', 'psro', 'tough']
  |S|=3: ['ef1_bargainer', 'mappo', 'openai_5.2_low']
  |S|=3: ['mappo', 'openai_5.2_low', 'psro']
  |S|=3: ['mappo', 'ppo', 'psro']
  |S|=4: ['ppo', 'psro', 'soft', 'tough']
  |S|=4: ['mappo', 'ppo', 'psro', 'tough']
  |S|=4: ['ef1_bargainer', 'mappo', 'openai_5.2_low', 'psro']
  |S|=4: ['mappo', 'openai_5.2_low', 'ppo', 'psro']
  |S|=5: ['mappo', 'ppo', 'psro', 'soft', 'tough']
  |S|=5: ['mappo', 'openai_5.2_low', 'ppo', 'psro', 'tough']
  |S|=5: ['mappo', 'openai_5.2_low', 'openai_5.2_none', 'ppo', 'psro']
  |S|=5: ['ef1_bargainer', 'mappo', 'openai_5.2_low', 'ppo', 'psro']
  |S|=5: ['mappo', 'nfsp', 'openai_5.2_low', 'ppo', 'psro']
  |S|=6: ['mappo', 'openai_5.2_low', 'ppo', 'psro', 'soft', 'tough']
  |S|=6: ['mappo', 'openai_5.2_low', 'openai_5.2_none', 'ppo', 'psro', 'tough']
  |S

In [9]:
solution_concept = "mene"
mg_full = MetaGame(policies=strategy_names, payoff_matrix=avg_payoff)
sigma_full = mg_full.solve(solution_concept)
sigma_full

array([-0.        , -0.        , -0.        , -0.        , -0.        ,
       -0.        , -0.        ,  0.79623596,  0.20376404, -0.        ])

In [14]:
from numpy.linalg import norm
# payoff distance between PPO and PSRO columns
ppo_idx = strategy_names.index("ppo")
psro_idx = strategy_names.index("psro")
print(norm(avg_payoff[:, ppo_idx] - avg_payoff[:, psro_idx]))



36.755057500281524


In [15]:
from numpy.linalg import norm                                                                       
                
print(f"{'':>12}", end="")                                                                          
for name in strategy_names:
    print(f" {name[:8]:>8}", end="")
print()

for i, ni in enumerate(strategy_names):
    print(f"{ni:>12}", end="")
    for j, nj in enumerate(strategy_names):
        d = norm(avg_payoff[:, i] - avg_payoff[:, j])
        print(f" {d:>8.1f}", end="")
    print()

                 walk    tough     soft openai_5 openai_5 ef1_barg     nfsp      ppo     psro    mappo
        walk      0.0    253.5    468.6    208.6    219.0    165.3    149.0    184.8    196.7    212.7
       tough    253.5      0.0    533.6    267.2    274.5    309.5    280.4    211.9    182.1    166.0
        soft    468.6    533.6      0.0    322.0    318.1    353.4    348.2    363.2    382.3    391.0
openai_5.2_none    208.6    267.2    322.0      0.0     23.3     90.4     74.5     57.3     89.1    109.3
openai_5.2_low    219.0    274.5    318.1     23.3      0.0     97.3     85.0     64.9     94.5    113.1
ef1_bargainer    165.3    309.5    353.4     90.4     97.3      0.0     38.7    118.2    151.4    175.6
        nfsp    149.0    280.4    348.2     74.5     85.0     38.7      0.0     91.5    124.0    148.3
         ppo    184.8    211.9    363.2     57.3     64.9    118.2     91.5      0.0     36.8     60.1
        psro    196.7    182.1    382.3     89.1     94.5    151.4 

In [10]:
mg_full = MetaGame(policies=strategy_names, payoff_matrix=avg_payoff)
sigma_full = mg_full.solve(solution_concept)
full_metrics = {"uw": float(sigma_full @ avg_payoff @ sigma_full)}
for m in METRIC_KEYS:
    M = np.nan_to_num(matrices[m], nan=0.0)
    full_metrics[m] = float(sigma_full @ M @ sigma_full)

print(f"\nFull game support: {[strategy_names[i] for i, w in enumerate(sigma_full) if w > 1e-2]}")
print(f"Full game metrics: { {k: f'{v:.4f}' for k, v in full_metrics.items()} }")




Full game support: ['ppo', 'psro']
Full game metrics: {'uw': '367.3338', 'nw': '325.9095', 'nw_plus': '51.2442', 'ef1': '0.5123', 'ef1_plus': '0.5293'}


In [11]:
all_metrics = ["uw"] + list(METRIC_KEYS)
header = f"{'CURB Set':<40} {'|S|':>4}"
for m in all_metrics:
    header += f" {m:>10} {'Δ'+m:>10}"
header += f" {'Min?':>5}"
print(f"\n{header}")
print("-" * len(header))

results = []
for S in sorted(all_curb, key=len):
    idx = sorted(S)
    sub_names = [strategy_names[i] for i in idx]

    #if not any(item in sub_names for item in ["ppo", "psro"]):
    sub_payoff = avg_payoff[np.ix_(idx, idx)]

    mg_sub = MetaGame(policies=sub_names, payoff_matrix=sub_payoff)
    sigma_sub = mg_sub.solve(solution_concept)

    row = {"curb_set": S, "size": len(S), "minimal": S in minimal_curb}
    row["uw"] = float(sigma_sub @ sub_payoff @ sigma_sub)
    for m in METRIC_KEYS:
        M_sub = np.nan_to_num(matrices[m][np.ix_(idx, idx)], nan=0.0)
        row[m] = float(sigma_sub @ M_sub @ sigma_sub)

    # Deltas
    for m in all_metrics:
        row[f"delta_{m}"] = row[m] - full_metrics[m]

    results.append(row)

    label = "{" + ", ".join(sub_names[:3])
    if len(sub_names) > 3:
        label += f", +{len(sub_names)-3}"
    label += "}"
    is_min = "*" if row["minimal"] else ""

    line = f"{label:<40} {row['size']:>4}"
    for m in all_metrics:
        line += f" {row[m]:>10.4f} {row[f'delta_{m}']:>+10.4f}"
    line += f" {is_min:>5}"
    print(line)



CURB Set                                  |S|         uw        Δuw         nw        Δnw    nw_plus   Δnw_plus        ef1       Δef1   ef1_plus  Δef1_plus  Min?
-----------------------------------------------------------------------------------------------------------------------------------------------------------------
{ppo}                                       1   368.9641    +1.6304   327.8616    +1.9521    52.6599    +1.4157     0.5235    +0.0112     0.5418    +0.0125     *
{psro}                                      1   365.9879    -1.3458   322.6325    -3.2769    51.6788    +0.4347     0.5336    +0.0213     0.5524    +0.0230     *
{mappo}                                     1   364.8781    -2.4556   322.6952    -3.2143    49.7178    -1.5264     0.4571    -0.0552     0.5135    -0.0159     *
{tough, ppo}                                2   368.9641    +1.6304   327.8616    +1.9521    52.6599    +1.4157     0.5235    +0.0112     0.5418    +0.0125      
{openai_5.2_low, mappo}    

In [12]:
sigma_full = MetaGame(policies=strategy_names, payoff_matrix=avg_payoff).solve(solution_concept)    
support_str = ", ".join(                                                                            
    f"{strategy_names[i]}:{sigma_full[i]:.1%}"
    for i in range(len(strategy_names)) if sigma_full[i] > 0.01
)
print(f"  Equilibrium: {support_str}")
print("=" * len(header))

  Equilibrium: ppo:79.6%, psro:20.4%


In [ ]:
for i, name in enumerate(strategy_names):                                                           
      sigma_pure = np.zeros(len(strategy_names))                                                      
      sigma_pure[i] = 1.0                                                                             
                  
      expected_utils = avg_payoff @ sigma_pure  # payoff of each strategy vs pure i                   
      self_payoff = avg_payoff[i, i]            # payoff of i vs i

      regret = expected_utils - self_payoff
      max_regret = np.max(regret)
      best_response = strategy_names[np.argmax(expected_utils)]

      is_ne = "✓ NE" if max_regret <= 1e-3 else ""
      print(f"{name:>20}: max_regret={max_regret:>+10.4f}  BR={best_response:<20} {is_ne}")

                walk: max_regret=   +0.0000  BR=walk                 ✓ NE
               tough: max_regret=   +2.0409  BR=ppo                  
                soft: max_regret= +275.5792  BR=tough                
     openai_5.2_none: max_regret=   +1.8138  BR=openai_5.2_low       
      openai_5.2_low: max_regret=   +9.7375  BR=mappo                
       ef1_bargainer: max_regret=  +15.1498  BR=openai_5.2_low       
                nfsp: max_regret=  +12.2069  BR=openai_5.2_low       
                 ppo: max_regret=   +0.0000  BR=ppo                  ✓ NE
                psro: max_regret=   +0.0000  BR=psro                 ✓ NE
               mappo: max_regret=   +0.0000  BR=mappo                ✓ NE


('ppo', 'psro')                     σ=(ppo:79.6%, psro:20.4%         )  max_regret= +0.0000 ✓ NE
('ppo', 'mappo')                    σ=(ppo:25.7%, mappo:74.3%        )  max_regret= +0.7720 
('psro', 'mappo')                   σ=(psro:19.8%, mappo:80.2%       )  max_regret= +0.0000 ✓ NE
('ppo', 'psro', 'mappo')            σ=(ppo:79.6%, psro:20.4%         )  max_regret= +0.0000 ✓ NE


In [13]:
 
short_names = [SHORT_MAP.get(s, s) for s in strategy_names]
strat_order = [short_names.index(s) for s in STRAT_DISPLAY_ORDER if s in short_names]

# --- Build matrices ---
# Strategy membership: (n_eco, n_strats)
mem_matrix = np.zeros((n_eco, len(strat_order)))
for row, r in enumerate(results_sorted):
    for col, si in enumerate(strat_order):
        if si in r['curb_set']:
            mem_matrix[row, col] = 1.0

# Delta matrix: (n_eco, n_metrics)
delta_matrix = np.zeros((n_eco, len(all_metrics)))
for row, r in enumerate(results_sorted):
    for col, m in enumerate(all_metrics):
        delta_matrix[row, col] = r[f'delta_{m}']

# % change from full game
pct_matrix = np.zeros_like(delta_matrix)
for col, m in enumerate(all_metrics):
    baseline = full_metrics[m]
    if abs(baseline) > 1e-10:
        pct_matrix[:, col] = delta_matrix[:, col] / abs(baseline) * 100

# --- Y labels ---
y_labels = []
for r in results_sorted:
    idx = sorted(r['curb_set'])
    names = [SHORT_MAP.get(strategy_names[i], strategy_names[i]) for i in idx]
    if len(names) <= 4:
        label = '{' + ', '.join(names) + '}'
    else:
        label = '{' + ', '.join(names[:3]) + f', +{len(names)-3}' + '}'
    marker = ' *' if r['minimal'] else ''
    y_labels.append(f'{label}{marker}')

# --- Figure: [membership | % change heatmap] ---
fig = plt.figure(figsize=(12, 0.45 * n_eco + 2), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.2], wspace=0.03)

y_positions = np.arange(n_eco)

# Panel A: Strategy membership
ax_mem = fig.add_subplot(gs[0, 0])
mem_rgb = np.ones((n_eco, len(strat_order), 4))
for row in range(n_eco):
    for col, si in enumerate(strat_order):
        if mem_matrix[row, col] > 0:
            sname = short_names[si]
            mem_rgb[row, col] = mcolors.to_rgba(STRAT_COLORS.get(sname, '#333333'))
        else:
            mem_rgb[row, col] = (0.95, 0.95, 0.95, 1.0)

ax_mem.imshow(mem_rgb, aspect='auto', interpolation='nearest')
ax_mem.set_xticks(range(len(strat_order)))
ax_mem.set_xticklabels(
    [STRAT_DISPLAY_ORDER[i] for i in range(len(strat_order))],
    fontsize=7, rotation=55, ha='right',
)
ax_mem.set_yticks(y_positions)
ax_mem.set_yticklabels(y_labels, fontsize=7, fontfamily='monospace')
ax_mem.set_title('Strategy membership', fontsize=9, fontweight='bold')
ax_mem.tick_params(axis='both', length=0)
for y in np.arange(-0.5, n_eco, 1):
    ax_mem.axhline(y, color='white', linewidth=0.5)
for x in np.arange(-0.5, len(strat_order), 1):
    ax_mem.axvline(x, color='white', linewidth=0.5)

# Panel B: % change heatmap
ax_heat = fig.add_subplot(gs[0, 1])
cmap = plt.cm.RdYlBu.copy()
pct_masked = np.ma.masked_invalid(pct_matrix)
vabs = max(3, np.nanmax(np.abs(pct_matrix)))
im = ax_heat.imshow(pct_masked, aspect='auto', cmap=cmap,
                    vmin=-vabs, vmax=vabs, interpolation='nearest')

for row in range(n_eco):
    for col in range(len(all_metrics)):
        val = pct_matrix[row, col]
        text_color = 'white' if abs(val) > vabs * 0.6 else 'black'
        ax_heat.text(col, row, f'{val:+.1f}%', ha='center', va='center',
                    fontsize=6, color=text_color, fontweight='bold')

ax_heat.set_xticks(range(len(all_metrics)))
ax_heat.set_xticklabels(METRIC_LABELS, fontsize=9, fontweight='bold')
ax_heat.set_yticks([])
ax_heat.set_title('% change from full game', fontsize=9, fontweight='bold')
ax_heat.tick_params(axis='both', length=0)
for y in np.arange(-0.5, n_eco, 1):
    ax_heat.axhline(y, color='white', linewidth=1)
for x in np.arange(-0.5, len(all_metrics), 1):
    ax_heat.axvline(x, color='white', linewidth=1)

cbar = fig.colorbar(im, ax=ax_heat, fraction=0.04, pad=0.02)
cbar.set_label('% change from full game', fontsize=8)
cbar.ax.tick_params(labelsize=7)

fig.suptitle(
    'CURB Set Welfare Landscape (* = minimal)',
    fontsize=11, fontweight='bold', y=1.02,
)
plt.savefig('curb_welfare_landscape.png', dpi=150, bbox_inches='tight')
plt.show()


NameError: name 'SHORT_MAP' is not defined

In [ ]:
fig, axes = plt.subplots(1, len(minimal_curb) + 1, figsize=(3.5 * (len(minimal_curb) + 1), 3.5),                                               
                           constrained_layout=True)
                                                                                                                                                 
# Colors by strategy type                                                                                                                      
colors = {'walk': '#bbbbbb', 'tough': '#bbbbbb', 'soft': '#bbbbbb',                                                                            
        'nfsp': '#555555', 'mappo': '#555555', 'ppo': '#555555', 'psro': '#555555',                                                          
        'openai_5.2_none': '#e6550d', 'openai_5.2_low': '#e6550d', 'ef1_bargainer': '#e6550d'}                                               
                                                                                                                                                
# Full game first                                                                                                                              
ax = axes[0]                                                                                                                                   
bar_colors = [colors.get(s, '#333333') for s in strategy_names]
ax.bar(range(n), sigma_full, color=bar_colors, edgecolor='#333333', linewidth=0.5)
ax.set_xticks(range(n))
ax.set_xticklabels([s[:8] for s in strategy_names], fontsize=6, rotation=55, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Equilibrium weight')
ax.set_title(f'Full game (n={n})', fontsize=9, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Each minimal CURB set
for k, S in enumerate(sorted(minimal_curb, key=len)):
    ax = axes[k + 1]
    idx = sorted(S)
    sub_names = [strategy_names[i] for i in idx]
    sub_payoff = avg_payoff[np.ix_(idx, idx)]
    mg_sub = MetaGame(policies=sub_names, payoff_matrix=sub_payoff)
    sigma_sub = mg_sub.solve(solution_concept)

    # Map to full strategy list
    weights = np.zeros(n)
    for local_i, global_i in enumerate(idx):
        weights[global_i] = sigma_sub[local_i]

    in_set = [i in S for i in range(n)]
    bar_colors = [colors.get(s, '#333333') if in_set[i] else '#f0f0f0' for i, s in enumerate(strategy_names)]
    edge_colors = ['#333333' if in_set[i] else '#cccccc' for i in range(n)]

    ax.bar(range(n), weights, color=bar_colors, edgecolor=edge_colors, linewidth=0.5)
    ax.set_xticks(range(n))
    ax.set_xticklabels([s[:8] for s in strategy_names], fontsize=6, rotation=55, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_title('{' + ', '.join(sub_names[:3]) + '}', fontsize=9, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Equilibrium Support: Full Game vs Minimal CURB Sets', fontsize=12, fontweight='bold')
plt.savefig('curb_support_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_31543/1821765917.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
from evaluation.curb_analysis import compute_cbr

# Compute CBR for each singleton
cbr_map = {}
for i in range(n):
    cbr = compute_cbr(avg_payoff, frozenset({i}))
    cbr_map[strategy_names[i]] = [strategy_names[j] for j in cbr]

# Build adjacency matrix: cbr_adj[i,j] = 1 if j is in CBR({i})
cbr_adj = np.zeros((n, n))
for i in range(n):
    cbr = compute_cbr(avg_payoff, frozenset({i}))
    for j in cbr:
        cbr_adj[i, j] = 1.0

fig, ax = plt.subplots(figsize=(8, 7))
cmap = mcolors.ListedColormap(['#f0f0f0', '#2171b5'])
ax.imshow(cbr_adj, cmap=cmap, interpolation='nearest')

# Labels
short = [s[:8] for s in strategy_names]
ax.set_xticks(range(n))
ax.set_xticklabels(short, fontsize=8, rotation=45, ha='right')
ax.set_yticks(range(n))
ax.set_yticklabels(short, fontsize=8)
ax.set_xlabel('Strategy j (is j a BR to some mixture over {i}?)', fontsize=9)
ax.set_ylabel('Singleton set {i}', fontsize=9)
ax.set_title('CBR Adjacency: CBR({i}) for each strategy i', fontsize=11, fontweight='bold')

# Annotate
for i in range(n):
    for j in range(n):
        if cbr_adj[i, j] == 1:
            ax.text(j, i, '✓', ha='center', va='center', fontsize=10, color='white')

# Grid
for y in np.arange(-0.5, n, 1):
    ax.axhline(y, color='white', linewidth=1)
for x in np.arange(-0.5, n, 1):
    ax.axvline(x, color='white', linewidth=1)

# Highlight self-loops (minimal CURB singletons)
for S in minimal_curb:
    if len(S) == 1:
        i = list(S)[0]
        ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False,
                                    edgecolor='red', linewidth=2.5))

ax.legend(handles=[plt.Rectangle((0,0),1,1, fc='red', fill=False, linewidth=2.5, label='Minimal CURB singleton')],
        loc='lower right', fontsize=8)

plt.savefig('cbr_adjacency.png', dpi=200, bbox_inches='tight')
plt.show()


/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_31543/3441473879.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
sorted_curb = sorted(all_curb, key=len)                                                                                                        
                                                                                                                                                
# Find Hasse edges (direct cover relations)
hasse_edges = []                                                                                                                               
for i, S in enumerate(sorted_curb):
    for j, T in enumerate(sorted_curb):
        if S < T:
            has_intermediate = any(S < U < T for U in sorted_curb)
            if not has_intermediate:
                hasse_edges.append((i, j))

# Assign positions
from collections import defaultdict
levels = defaultdict(list)
for i, S in enumerate(sorted_curb):
    levels[len(S)].append(i)

pos = {}
for size, indices in levels.items():
    width = len(indices)
    for k, idx in enumerate(indices):
        x = (k - (width - 1) / 2) * 1.8
        pos[idx] = (x, size)

# Assign a color to each node based on which minimal CURB set(s) it contains
minimal_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#9467bd', '#8c564b', '#e377c2']
min_list = sorted(minimal_curb, key=lambda s: sorted(s))

def node_color(S):
    """Color by the first minimal CURB set contained in S."""
    for k, m in enumerate(min_list):
        if m.issubset(S):
            return minimal_colors[k % len(minimal_colors)]
    return '#999999'

fig, ax = plt.subplots(figsize=(18, 10))

# Draw directed arrows colored by source node
for i, j in hasse_edges:
      x0, y0 = pos[i]
      x1, y1 = pos[j]
      ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                  arrowprops=dict(arrowstyle='->', color='#999999', lw=1.0, alpha=0.6,
                                  shrinkA=6, shrinkB=6))

# Draw nodes

def draw_pie_node(fig, ax, x, y, colors, size_px=14):                                                                                                                  
      """Place a tiny pie chart at data coords (x, y)."""                                                                                                                
      # Convert data coords to figure fraction                                                                                                                           
      disp = ax.transData.transform((x, y))                                                                                                                              
      fig_coord = fig.transFigure.inverted().transform(disp)                                                                                                             
      # size in figure fraction                                                                                                                                          
      sz = size_px / fig.dpi  # convert pixels to inches, then to figure fraction                                                                                        
      sz_frac = sz / fig.get_size_inches()[0]  # approximate                                                                                                             
                                                                                                                                                                         
      ax_inset = fig.add_axes(                                                                                                                                           
          [fig_coord[0] - sz_frac/2, fig_coord[1] - sz_frac/2, sz_frac, sz_frac],
          zorder=2
      )
      ax_inset.set_aspect('equal')
      ax_inset.pie([1]*len(colors), colors=colors,
                   wedgeprops=dict(edgecolor='grey', linewidth=0.8))
      ax_inset.set_frame_on(False)

for i, S in enumerate(sorted_curb):                                                                                                                                    
      x, y = pos[i]                                                                                                                                                      
      is_min = S in minimal_curb                                                                                                                                         
                                                                                                                                                                         
      contained = [k for k, m in enumerate(min_list) if m.issubset(S)]                                                                                                   
      colors = [minimal_colors[k % len(minimal_colors)] for k in contained]                                                                                              
      if not colors:                                                                                                                                                     
          colors = ["#000000"]                                                                                                                                           
                                                                                                                                                                         
      outer_size = 180 if is_min else 100                                                                                                                                
                                                                                                                                                                         
      # Outer ring: first minimal
      ax.scatter(x, y, s=outer_size, c=colors[0], zorder=2,
                 edgecolors='grey', linewidth=0.8)

      # Inner dot: second minimal (if present)
      if len(colors) >= 2:
          ax.scatter(x, y, s=outer_size * 0.35, c=colors[1], zorder=3,
                     edgecolors='grey', linewidth=0.5)

      # Third minimal as tiny center dot (rare)
      if len(colors) >= 3:
          ax.scatter(x, y, s=outer_size * 0.12, c=colors[2], zorder=4,
                     edgecolors='none')

      # Labels
      if is_min:
          names = sorted([strategy_names[j][:6] for j in S])
          label = '{' + ','.join(names) + '}'
          ax.annotate(label, (x, y), textcoords="offset points", xytext=(0, 12),
                      fontsize=6, ha='center', fontweight='bold')
      elif len(S) == n:
          ax.annotate('Full', (x, y), textcoords="offset points", xytext=(0, 12),
                      fontsize=7, ha='center', fontweight='bold')
      else:
          idx_str = ','.join(str(j) for j in sorted(S))
          ax.text(x, y, idx_str, fontsize=4, ha='center', va='center',
                  color='black', fontweight='bold', zorder=5)

ax.set_ylabel('CURB set size |S|', fontsize=10)
ax.set_yticks(range(1, n + 1))
ax.set_title('CURB Set Lattice (Hasse Diagram)', fontsize=12, fontweight='bold')
ax.set_xticks([])

# Legend: map minimal CURB sets to colors
from matplotlib.lines import Line2D
legend_handles = []
for k, m in enumerate(min_list):
    names = sorted([strategy_names[j][:6] for j in m])
    label = '{' + ','.join(names) + '}'
    legend_handles.append(
        Line2D([0], [0], marker='o', color='w', markerfacecolor=minimal_colors[k % len(minimal_colors)],
                markersize=10, label=label))
ax.legend(handles=legend_handles, loc='upper left', fontsize=8, title='Minimal CURB sets')

# Index key at bottom
index_key = '  '.join(f'{i}={strategy_names[i][:8]}' for i in range(n))
ax.text(0.5, -0.06, f'Index key: {index_key}', transform=ax.transAxes,
        fontsize=7, ha='center', fontfamily='monospace')

plt.savefig('curb_lattice.png', dpi=200, bbox_inches='tight')
plt.show()

/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_31543/2752495747.py:127: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
from src.iterative_game_analysis.solvers.lle import (
    construct_player_kernel, _payoff_tensor_from_matrix, affinity_entropy
)

SHORT = {
    'walk': 'Walk', 'tough': 'Tough', 'soft': 'Soft',
    'nfsp': 'NFSP', 'mappo': 'MAPPO', 'ppo': 'PPO', 'psro': 'PSRO',
    'openai_5.2_none': '5.2:N', 'openai_5.2_low': '5.2:L', 'ef1_bargainer': 'EF1',
}
showcase_curbs = []
target_sizes = [2, 3, 5, 7]
for sz in target_sizes:
    for S in sorted(all_curb, key=lambda s: sorted(s)):
        if len(S) == sz and S not in showcase_curbs:
            showcase_curbs.append(S)
            break
showcase_curbs.append(frozenset(range(n)))  # full game

n_panels = len(showcase_curbs)
fig, axes = plt.subplots(1, n_panels, figsize=(4.2 * n_panels, 4.0),
                         constrained_layout=True)

var = 300  

for ax, S in zip(axes, showcase_curbs):
    idx = sorted(S)
    sub_payoff = avg_payoff[np.ix_(idx, idx)]
    pt = _payoff_tensor_from_matrix(sub_payoff)
    kernel = construct_player_kernel(pt, player=0, var=var)
    sub_names = [SHORT.get(strategy_names[i], strategy_names[i]) for i in idx]
    k = len(idx)
    
    im = ax.imshow(kernel, cmap='YlOrRd', vmin=0, vmax=1, interpolation='nearest')
    
    for r in range(k):
        for c in range(k):
            v = kernel[r, c]
            color = 'white' if v > 0.65 else 'black'
            ax.text(c, r, f'{v:.2f}', ha='center', va='center',
                    fontsize=max(6, 9 - k // 3), color=color, fontweight='bold')
    
    ax.set_xticks(range(k))
    ax.set_xticklabels(sub_names, fontsize=7, rotation=45, ha='right')
    ax.set_yticks(range(k))
    ax.set_yticklabels(sub_names, fontsize=7)
    for y in np.arange(-0.5, k, 1):
        ax.axhline(y, color='white', linewidth=0.5)
    for x in np.arange(-0.5, k, 1):
        ax.axvline(x, color='white', linewidth=0.5)
    
    is_min = S in minimal_curb
    is_full = len(S) == n
    if is_full:
        label = f'Full game (n={k})'
    elif k <= 5:
        label = '{' + ', '.join(sub_names) + '}'
    else:
        label = '{' + ', '.join(sub_names[:3]) + f', +{k-3}' + '}'
    if is_min:
        label += ' *'
    
    uniform = np.ones(k) / k
    h = affinity_entropy(uniform, kernel, p=1.0)
    label += f'\nH_aff(unif)={h:.3f}'
    ax.set_title(label, fontsize=8, fontweight='bold')

cbar = fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02)
cbar.set_label(r'$K_{ij} = \exp(-\|p_i - p_j\|^2\, /\, 2\sigma^2)$', fontsize=9)
cbar.ax.tick_params(labelsize=7)

fig.suptitle(
    f'Affinity Kernel Across CURB Sets ($\\sigma^2$={var})\n'
    'Strategy similarity reshapes as the restricted game changes (* = minimal)',
    fontsize=12, fontweight='bold', y=1.04
)

plt.savefig('affinity_kernel_curb_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_31543/85461560.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Multi-Metric Affinity Kernels

Build kernels from different matrices (Payoff, EF1, NW) to see how "clone" structure depends on what you measure. Walk and EF1 bargainer are close in payoff-kernel but should diverge in EF1-kernel.

In [ ]:
def build_kernel_from_matrix(metric_matrix, idx, var):
    """Build Gaussian kernel from an arbitrary metric matrix restricted to idx."""
    sub = metric_matrix[np.ix_(idx, idx)]
    k = len(idx)
    # Rows = strategies, Cols = opponents (same as payoff tensor player 0)
    diff = sub[:, None, :] - sub[None, :, :]
    dist = np.sqrt(np.sum(diff ** 2, axis=-1))
    return np.exp(-dist / (2 * var))

# Metrics to compare
metric_configs = {
    'Payoff (UW)': avg_payoff,
    'Nash Welfare': np.nan_to_num(nw_matrix, nan=0.0),
    'EF1': np.nan_to_num(ef1_matrix, nan=0.0),
}

# Pick 3 representative CURB sets: small, medium, full
kernel_showcase = []
for S in sorted(all_curb, key=len):
    if len(S) == 3 and S not in kernel_showcase:
        kernel_showcase.append(S)
        break
for S in sorted(all_curb, key=len):
    if len(S) == 5 and S not in kernel_showcase:
        kernel_showcase.append(S)
        break
kernel_showcase.append(frozenset(range(n)))

n_metrics = len(metric_configs)
n_curbs = len(kernel_showcase)
var = 200

fig, axes = plt.subplots(n_curbs, n_metrics, figsize=(4.5 * n_metrics, 4.0 * n_curbs),
                         constrained_layout=True)

for row, S in enumerate(kernel_showcase):
    idx = sorted(S)
    sub_names = [SHORT.get(strategy_names[i], strategy_names[i]) for i in idx]
    k = len(idx)
    
    for col, (metric_name, metric_mat) in enumerate(metric_configs.items()):
        ax = axes[row, col]
        
        # Scale var per metric: normalize by median distance in that metric
        sub = metric_mat[np.ix_(idx, idx)]
        diff = sub[:, None, :] - sub[None, :, :]
        dists = np.sqrt(np.sum(diff ** 2, axis=-1))
        off_diag = dists[~np.eye(k, dtype=bool)]
        # Use median distance to auto-scale var so kernel has good spread
        med_dist = np.median(off_diag) if len(off_diag) > 0 else 1.0
        auto_var = max(med_dist / (2 * np.log(2)), 1e-6)  # K=0.5 at median
        
        kernel = build_kernel_from_matrix(metric_mat, idx, auto_var)
        
        im = ax.imshow(kernel, cmap='YlOrRd', vmin=0, vmax=1, interpolation='nearest')
        for r in range(k):
            for c in range(k):
                v = kernel[r, c]
                color = 'white' if v > 0.65 else 'black'
                ax.text(c, r, f'{v:.2f}', ha='center', va='center',
                        fontsize=max(5, 8 - k // 3), color=color, fontweight='bold')
        
        ax.set_xticks(range(k))
        ax.set_xticklabels(sub_names, fontsize=6, rotation=45, ha='right')
        ax.set_yticks(range(k))
        ax.set_yticklabels(sub_names if col == 0 else [], fontsize=6)
        
        for y in np.arange(-0.5, k, 1):
            ax.axhline(y, color='white', linewidth=0.5)
        for x in np.arange(-0.5, k, 1):
            ax.axvline(x, color='white', linewidth=0.5)
        
        if row == 0:
            ax.set_title(f'{metric_name}\n(auto σ²={auto_var:.1f})', fontsize=9, fontweight='bold')
    
    # Row label
    if len(S) == n:
        row_label = f'Full game (n={k})'
    elif k <= 5:
        row_label = '{' + ', '.join(sub_names) + '}'
    else:
        row_label = '{' + ', '.join(sub_names[:3]) + f', +{k-3}' + '}'
    axes[row, 0].set_ylabel(row_label, fontsize=9, fontweight='bold')

cbar = fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02)
cbar.set_label('Kernel similarity K(i,j)', fontsize=9)
cbar.ax.tick_params(labelsize=7)

fig.suptitle(
    'Affinity Kernels by Metric: Who is a "clone" depends on what you measure',
    fontsize=13, fontweight='bold', y=1.02
)
plt.savefig('multi_metric_kernel_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_31543/1832865772.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
import cvxpy as cp
print(cp.__version__)
print(cp.installed_solvers())

1.8.1
['CLARABEL', 'CVXOPT', 'GLPK', 'GLPK_MI', 'HIGHS', 'OSQP', 'SCIPY', 'SCS']


## polarix Equilibrium Ratings & Contribution Analysis                                                                                                                    
                                                                                                                                                                            
For each minimal CURB set (and the full game), we use polarix to:                                                                                                         
1. Compute max-entropy correlated equilibrium ratings                                                                                                                     
2. Show which opponent matchups drive each strategy's rating  

In [ ]:
import polarix as plx
import jax.numpy as jnp

def make_symmetric_2p_game(payoff, names):
    strategies = np.array(names)
    payoffs = np.stack([payoff, payoff.T])
    return plx.Game(
        payoffs=payoffs,
        actions=(strategies, strategies),
        players=('row', 'column'),
        symmetry_groups=(0, 0),
    )


In [ ]:
print('This is the full a game ')
game_full = make_symmetric_2p_game(avg_payoff, strategy_names)
ce_full = plx.solve(game_full, plx.ce_maxent, max_num_iterations=100_000)
print('Converged:', ce_full.is_terminal())


This is the full a game 


Solving Game((2, 10, 10)):  23%|██▎       | 23000/100000 [00:01<00:04, 18596.36it/s, ['ce_gap']=7.6293945e-06, ['entropy']=1.9594449, ['loss']=1.9593492, ['me_loss']=1.959266, ['residual_norm']=7.6322385e-06, ['wd_loss']=83.21598]   


Converged: True


In [ ]:
print(f"CE ratings: {ce_full.ratings[0]}")

CE ratings: [-5.3209766e+01 -3.9840424e+01 -1.7001947e+02 -1.3469279e+01
 -6.0025949e+00 -1.6064083e+01 -1.3571442e+01 -1.2645276e+00
  1.3864535e-05 -2.5869129e+00]


In [ ]:
display(plx.plot_rating_and_marginal( game_full, ce_full.ratings, plx.marginals_from_joint(ce_full.joint), ))

alt.HConcatChart(...)

In [ ]:
lle_result = plx.solve(game_full, plx.lle, max_num_iterations=100_000)                                                                                                    

display(plx.plot_rating_and_marginal(                                                                                                                                     
    game_full, lle_result.ratings,
    lle_result.marginals,
))


Solving Game((2, 10, 10)):  26%|██▌       | 26000/100000 [00:02<00:07, 9823.08it/s, ['exp']=0.0, ['loss']=-0.00012207031, ['max_threshold_annealed']=0.00048828125, ['schedule_count']=42.0, ['temperature']=0.11598216, ['terminal']=True, ['threshold']=-0.00012207031, ['threshold_count']=112.0, ['trigger']=-0.00012207031]            


alt.HConcatChart(...)

In [ ]:
lle_result = plx.solve(game_full, plx.lle, max_num_iterations=100_000)

joint = plx.joint_from_marginals(lle_result.marginals)
plx.plot_rating_contribution(game_full, joint, rating_player=0, contrib_player=1)


Solving Game((2, 10, 10)):  26%|██▌       | 26000/100000 [00:02<00:07, 10495.11it/s, ['exp']=0.0, ['loss']=-0.00012207031, ['max_threshold_annealed']=0.00048828125, ['schedule_count']=42.0, ['temperature']=0.11598216, ['terminal']=True, ['threshold']=-0.00012207031, ['threshold_count']=112.0, ['trigger']=-0.00012207031]           


alt.HConcatChart(...)

In [ ]:
display(plx.plot_rating_contribution(
    game_full, ce_full.joint,
    rating_player=0,
    contrib_player=1,
))

alt.HConcatChart(...)

In [ ]:
for metric_name, metric_matrix in [
    ('uw (payoff)', avg_payoff),                                                                                                                                          
    ('nw', matrices['nw']),
    ('nw_plus', matrices['nw_plus']),                                                                                                                                     
    ('ef1', matrices['ef1']),
    ('ef1_plus', matrices['ef1_plus']),
]:
    M = np.nan_to_num(metric_matrix, nan=0.0)
    metric_game = make_symmetric_2p_game(M, strategy_names)

    print(f'\n=== {metric_name} contribution (full game CCE) ===')
    display(plx.plot_rating_contribution(
        metric_game, ce_full.joint,
        rating_player=0,
        contrib_player=1,
    ))




=== uw (payoff) contribution (full game CCE) ===


alt.HConcatChart(...)


=== nw contribution (full game CCE) ===


alt.HConcatChart(...)


=== nw_plus contribution (full game CCE) ===


alt.HConcatChart(...)


=== ef1 contribution (full game CCE) ===


alt.HConcatChart(...)


=== ef1_plus contribution (full game CCE) ===


alt.HConcatChart(...)

In [ ]:
for S in sorted(all_curb, key=len):
    idx = sorted(S)
    sub_names = [strategy_names[i] for i in idx]
    sub_payoff = avg_payoff[np.ix_(idx, idx)]

    label = ', '.join(sub_names)
    if len(label) > 60:
        label = label[:57] + '...'
    print(f'\n=== MINIMAL CURB: {{{label}}} (|S|={len(S)}) ===')

    game_sub = make_symmetric_2p_game(sub_payoff, sub_names)
    ce_sub = plx.solve(game_sub, plx.ce_maxent, max_num_iterations=1_000_000)
    print('Converged:', ce_sub.is_terminal())

    display(plx.plot_rating_and_marginal(
        game_sub, ce_sub.ratings,
        plx.marginals_from_joint(ce_sub.joint),
    ))

    display(plx.plot_rating_contribution(
        game_sub, ce_sub.joint,
        rating_player=0,
        contrib_player=1,
    ))


=== MINIMAL CURB: {ppo} (|S|=1) ===


Solving Game((2, 1, 1)):   0%|          | 1000/1000000 [00:00<01:07, 14842.79it/s, ['ce_dual_per_player'][0]=[[0.]], ['ce_dual_per_player'][1]=[[0.]], ['ce_gap']=0.0, ['entropy']=-0.0, ['expected_ce_gain_per_player'][0]=[[-0.]], ['expected_ce_gain_per_player'][1]=[[-0.]], ['joint']=[[1.]], ['loss']=0.0, ['me_loss']=0.0, ['rating_per_player'][0]=[-0.], ['rating_per_player'][1]=[-0.], ['residual_norm']=0.0, ['wd_loss']=0.0]

Converged: True


alt.HConcatChart(...)

alt.HConcatChart(...)

...


=== MINIMAL CURB: {walk, tough, soft, openai_5.2_none, openai_5.2_low, ef1_b...} (|S|=10) ===


Solving Game((2, 10, 10)):   2%|▏         | 23000/1000000 [00:00<00:39, 24642.13it/s, ['ce_gap']=7.6293945e-06, ['entropy']=1.9594449, ['loss']=1.9593492, ['me_loss']=1.959266, ['residual_norm']=7.6322385e-06, ['wd_loss']=83.21598]   


Converged: True


alt.HConcatChart(...)

alt.HConcatChart(...)

In [ ]:
for S in sorted(all_curb, key=len):
    if len(S) < 2 or len(S) == n:
        continue

    idx = sorted(S)
    sub_names = [strategy_names[i] for i in idx]
    sub_payoff = avg_payoff[np.ix_(idx, idx)]

    label = ', '.join(sub_names)
    if len(label) > 60:
        label = label[:57] + '...'
    print(f'\n=== CURB: {{{label}}} (|S|={len(S)}) ===')

    game_sub = make_symmetric_2p_game(sub_payoff, sub_names)
    ce_sub = plx.solve(game_sub, plx.ce_maxent, max_num_iterations=100_000)

    display(plx.plot_rating_and_marginal(
        game_sub, ce_sub.ratings,
        plx.marginals_from_joint(ce_sub.joint),
    ))
    display(plx.plot_rating_contribution(
        game_sub, ce_sub.joint,
        rating_player=0, contrib_player=1,
    ))


=== CURB: {tough, ppo} (|S|=2) ===


Solving Game((2, 2, 2)):  22%|██▏       | 22000/100000 [00:00<00:02, 34155.72it/s, ['ce_gap']=8.978881e-06, ['entropy']=5.867346e-05, ['loss']=1.0454075e-05, ['me_loss']=4.410734e-06, ['rating_per_player'][0]=[-4.9456787e+01  8.9788809e-06], ['rating_per_player'][1]=[-4.9456787e+01  8.9788809e-06], ['residual_norm']=7.978881e-06, ['wd_loss']=6.043341]        


alt.HConcatChart(...)

alt.HConcatChart(...)


=== CURB: {openai_5.2_low, mappo} (|S|=2) ===


...

alt.HConcatChart(...)


=== CURB: {tough, soft, openai_5.2_none, openai_5.2_low, ef1_bargain...} (|S|=9) ===


Solving Game((2, 9, 9)):  44%|████▍     | 44000/100000 [00:01<00:01, 28331.14it/s, ['ce_gap']=1.0005897e-06, ['entropy']=1.8074293, ['loss']=1.8074237, ['me_loss']=1.8073566, ['residual_norm']=1.4142167e-06, ['wd_loss']=67.11141] 


alt.HConcatChart(...)

alt.HConcatChart(...)

In [ ]:
for S in sorted(all_curb, key=len):
    if len(S) < 2:
        continue                                                                                                                                                          

    idx = sorted(S)                                                                                                                                                       
    sub_names = [strategy_names[i] for i in idx]
    sub_payoff = avg_payoff[np.ix_(idx, idx)]

    label = ', '.join(sub_names)
    if len(label) > 60:
        label = label[:57] + '...'
    print(f'\n=== CURB: {{{label}}} (|S|={len(S)}) ===')

    # Solve CCE once on payoff matrix
    game_sub = make_symmetric_2p_game(sub_payoff, sub_names)
    ce_sub = plx.solve(game_sub, plx.ce_maxent, max_num_iterations=1_000_000)
    print('Converged:', ce_sub.is_terminal())

    display(plx.plot_rating_and_marginal(
        game_sub, ce_sub.ratings,
        plx.marginals_from_joint(ce_sub.joint),
    ))

    # Contribution for each metric using the same CCE joint
    for metric_name, full_matrix in [
        ('uw (payoff)', avg_payoff),
        ('nw', matrices['nw']),
        ('nw_plus', matrices['nw_plus']),
        ('ef1', matrices['ef1']),
        ('ef1_plus', matrices['ef1_plus']),
    ]:
        M = np.nan_to_num(full_matrix[np.ix_(idx, idx)], nan=0.0)
        metric_game = make_symmetric_2p_game(M, sub_names)

        print(f'  --- {metric_name} contribution ---')
        display(plx.plot_rating_contribution(
            metric_game, ce_sub.joint,
            rating_player=0,
            contrib_player=1,
        ))


=== CURB: {tough, ppo} (|S|=2) ===


Solving Game((2, 2, 2)):   2%|▏         | 22000/1000000 [00:00<00:28, 34281.04it/s, ['ce_gap']=8.978881e-06, ['entropy']=5.867346e-05, ['loss']=1.0454075e-05, ['me_loss']=4.410734e-06, ['rating_per_player'][0]=[-4.9456787e+01  8.9788809e-06], ['rating_per_player'][1]=[-4.9456787e+01  8.9788809e-06], ['residual_norm']=7.978881e-06, ['wd_loss']=6.043341]        


Converged: True


alt.HConcatChart(...)

  --- uw (payoff) contribution ---


...

alt.HConcatChart(...)

  --- ef1 contribution ---


alt.HConcatChart(...)

  --- ef1_plus contribution ---


alt.HConcatChart(...)